<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/01_database_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
HAM_ROOT = PROJECT_ROOT / 'Datasets' / 'HAM10000'
PAD_ROOT = PROJECT_ROOT / 'Datasets' / 'PAD-UFES-20'
DB_ROOT = PROJECT_ROOT / 'Database'
MODEL_ROOT = PROJECT_ROOT / 'Models'

for folder in (HAM_ROOT, PAD_ROOT, DB_ROOT, MODEL_ROOT):
    folder.mkdir(parents=True, exist_ok=True)

# Compatibility with your existing cells that concatenate path strings.
PROJECT_PATH = str(PROJECT_ROOT) + '/'
HAM_PATH = str(HAM_ROOT) + '/'
PAD_PATH = str(PAD_ROOT) + '/'
DATABASE_PATH = str(DB_ROOT / 'skin_cancer.db')
MODELS_PATH = str(MODEL_ROOT) + '/'

for name, value in {
    'Project': PROJECT_PATH,
    'HAM10000': HAM_PATH,
    'PAD-UFES-20': PAD_PATH,
    'Database file (to be created later)': DATABASE_PATH,
    'Models': MODELS_PATH,
}.items():
    print(f'{name}: {value}')

Project: /content/drive/MyDrive/cnn-skin-cancer-classification/
HAM10000: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/
PAD-UFES-20: /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/PAD-UFES-20/
Database file (to be created later): /content/drive/MyDrive/cnn-skin-cancer-classification/Database/skin_cancer.db
Models: /content/drive/MyDrive/cnn-skin-cancer-classification/Models/


In [4]:
probe = PROJECT_ROOT / '_drive_write_check.txt'
if probe.exists():
    raise FileExistsError('A previous check file exists; inspect it before rerunning.')
probe.write_text('Drive write check successful', encoding='utf-8')
assert probe.read_text(encoding='utf-8') == 'Drive write check successful'
print('Drive read/write check passed:', probe)

Drive read/write check passed: /content/drive/MyDrive/cnn-skin-cancer-classification/_drive_write_check.txt


In [5]:
probe.unlink()
print('Temporary check file removed.')

Temporary check file removed.


In [6]:
import pandas as pd
from IPython.display import display

IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
dataset_inventory = {}

for dataset_name, root in {'HAM10000': HAM_ROOT, 'PAD-UFES-20': PAD_ROOT}.items():
    files = [p for p in root.rglob('*') if p.is_file()]
    images = sorted(p for p in files if p.suffix.lower() in IMAGE_EXTENSIONS)
    csvs = sorted(p for p in files if p.suffix.lower() == '.csv')
    archives = [p for p in files if p.suffix.lower() in {'.zip', '.gz', '.tar'}]
    dataset_inventory[dataset_name] = {'images': images, 'csvs': csvs}
    print(f'\n{dataset_name}: {len(images)} image files; {len(csvs)} CSV files; '
          f'{len(archives)} archive files')
    print('CSV candidates:')
    for index, path in enumerate(csvs):
        print(index, path)
    print('First 3 images:')
    for path in images[:3]:
        print(path)
    if not images or not csvs:
        print('CHECK REQUIRED: extracted images and metadata CSV are both needed.')


HAM10000: 10033 image files; 5 CSV files; 0 archive files
CSV candidates:
0 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_metadata.csv
1 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_28_28_L.csv
2 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_28_28_RGB.csv
3 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_8_8_L.csv
4 /content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/hmnist_8_8_RGB.csv
First 3 images:
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024306.jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024307.jpg
/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_images_part_1/ISIC_0024308.jpg

PAD-UFES-20: 2300 image files; 1 CSV files; 0 archive files
CSV candidates:
0 /content/drive/MyDrive/cnn

In [7]:
ham_metadata_path = Path('/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/HAM10000/HAM10000_metadata.csv')
pad_metadata_path = Path('/content/drive/MyDrive/cnn-skin-cancer-classification/Datasets/PAD-UFES-20/metadata.csv')

for name, path in {'HAM10000': ham_metadata_path,
                   'PAD-UFES-20': pad_metadata_path}.items():
    if not path.is_file():
        raise FileNotFoundError(f'{name}: select an existing metadata CSV: {path}')
    frame = pd.read_csv(path)
    print(f'\n{name}: {frame.shape[0]} rows, {frame.shape[1]} columns')
    print('Columns:', frame.columns.tolist())
    display(frame.head())
    print('Data types:')
    print(frame.dtypes)


HAM10000: 10015 rows, 7 columns
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


Data types:
lesion_id        object
image_id         object
dx               object
dx_type          object
age             float64
sex              object
localization     object
dtype: object

PAD-UFES-20: 2298 rows, 26 columns
Columns: ['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']


,patient_id,lesion_id,smoke,drink,background_father,background_mother,age,pesticide,gender,skin_cancer_history,...,diameter_2,diagnostic,itch,grew,hurt,changed,bleed,elevation,img_id,biopsed
0,PAT_1516,1765,NaN,NaN,NaN,NaN,8,NaN,NaN,NaN,...,NaN,NEV,FALSE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1516_1765_530.png,False
1,PAT_46,881,False,False,POMERANIA,POMERANIA,55,False,FEMALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,TRUE,TRUE,TRUE,PAT_46_881_939.png,True
2,PAT_1545,1867,NaN,NaN,NaN,NaN,77,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1545_1867_547.png,False
3,PAT_1989,4061,NaN,NaN,NaN,NaN,75,NaN,NaN,NaN,...,NaN,ACK,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,PAT_1989_4061_934.png,False
4,PAT_684,1302,False,True,POMERANIA,POMERANIA,79,False,MALE,True,...,5.0,BCC,TRUE,TRUE,FALSE,FALSE,TRUE,TRUE,PAT_684_1302_588.png,True


Data types:
patient_id              object
lesion_id                int64
smoke                   object
drink                   object
background_father       object
background_mother       object
age                      int64
pesticide               object
gender                  object
skin_cancer_history     object
cancer_history          object
has_piped_water         object
has_sewage_system       object
fitspatrick            float64
region                  object
diameter_1             float64
diameter_2             float64
diagnostic              object
itch                    object
grew                    object
hurt                    object
changed                 object
bleed                   object
elevation               object
img_id                  object
biopsed                   bool
dtype: object


In [9]:
import pandas as pd

ham_df = pd.read_csv(ham_metadata_path)
pad_df = pd.read_csv(pad_metadata_path)

for name, df in {"HAM10000": ham_df, "PAD-UFES-20": pad_df}.items():
    print(f"\n{name}: {len(df)} rows")
    print("Columns:", df.columns.tolist())

    for column in ("dx", "diagnostic"):
        if column in df.columns:
            print(f"\nLabels in '{column}':")
            print(df[column].value_counts(dropna=False))


HAM10000: 10015 rows
Columns: ['lesion_id', 'image_id', 'dx', 'dx_type', 'age', 'sex', 'localization']

Labels in 'dx':
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64

PAD-UFES-20: 2298 rows
Columns: ['patient_id', 'lesion_id', 'smoke', 'drink', 'background_father', 'background_mother', 'age', 'pesticide', 'gender', 'skin_cancer_history', 'cancer_history', 'has_piped_water', 'has_sewage_system', 'fitspatrick', 'region', 'diameter_1', 'diameter_2', 'diagnostic', 'itch', 'grew', 'hurt', 'changed', 'bleed', 'elevation', 'img_id', 'biopsed']

Labels in 'diagnostic':
diagnostic
BCC    845
ACK    730
NEV    244
SEK    235
SCC    192
MEL     52
Name: count, dtype: int64


In [10]:
from pathlib import Path
import sqlite3
import pandas as pd
from IPython.display import display

# 1. Configuration
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/cnn-skin-cancer-classification"
)
HAM_ROOT = PROJECT_ROOT / "Datasets" / "HAM10000"
PAD_ROOT = PROJECT_ROOT / "Datasets" / "PAD-UFES-20"
DB_PATH = PROJECT_ROOT / "Database" / "skin_cancer.db"
DB_PATH.parent.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

# 2. Index actual image files, including nested folders
def build_image_index(root, use_stem=False):
    index = {}
    for path in root.rglob("*"):
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS:
            key = path.stem if use_stem else path.name
            if key in index:
                raise ValueError(
                    f"Duplicate image identifier: {key}\n"
                    f"{index[key]}\n{path}\n"
                    "Resolve these duplicate files before continuing."
                )
            index[key] = str(path)
    return index

print("Locating images. This may take a few minutes...")
ham_index = build_image_index(HAM_ROOT, use_stem=True)
pad_index = build_image_index(PAD_ROOT)

# 3. Select and standardise the three target labels
ham = ham_df.copy()
pad = pad_df.copy()

ham["dx"] = ham["dx"].astype("string").str.strip().str.lower()
pad["diagnostic"] = (
    pad["diagnostic"].astype("string").str.strip().str.upper()
)

ham = ham[ham["dx"].isin(["mel", "bcc"])].copy()
pad = pad[pad["diagnostic"].isin(["MEL", "BCC", "SCC"])].copy()

ham_records = pd.DataFrame({
    "dataset": "HAM10000",
    "image_id": ham["image_id"].astype("string").str.strip(),
    "lesion_id": ham["lesion_id"],
    "patient_id": None,
    "original_label": ham["dx"],
    "class_label": ham["dx"].map({"mel": "Melanoma", "bcc": "BCC"}),
    "age": pd.to_numeric(ham["age"], errors="coerce"),
    "sex": ham["sex"],
    "localization": ham["localization"],
})

pad_records = pd.DataFrame({
    "dataset": "PAD-UFES-20",
    "image_id": pad["img_id"].astype("string").str.strip(),
    "lesion_id": pad["lesion_id"],
    "patient_id": pad["patient_id"],
    "original_label": pad["diagnostic"],
    "class_label": pad["diagnostic"].map({
        "MEL": "Melanoma", "BCC": "BCC", "SCC": "SCC"
    }),
    "age": pd.to_numeric(pad["age"], errors="coerce"),
    "sex": pad["gender"],
    "localization": pad["region"],
})

ham_records["image_path"] = ham_records["image_id"].map(ham_index)
pad_records["image_path"] = pad_records["image_id"].map(pad_index)

records = pd.concat(
    [ham_records, pad_records], ignore_index=True
)

# 4. Check identifiers and image paths before creating the database
invalid_ids = (
    records["image_id"].isna() | records["image_id"].eq("")
)
if invalid_ids.any():
    display(records.loc[invalid_ids])
    raise ValueError("Missing image identifiers. No records imported.")

duplicates = records.duplicated(
    subset=["dataset", "image_id"], keep=False
)
if duplicates.any():
    display(records.loc[duplicates])
    raise ValueError("Duplicate metadata identifiers. No records imported.")

missing = records[records["image_path"].isna()]
if not missing.empty:
    display(missing[["dataset", "image_id", "class_label"]])
    raise FileNotFoundError(
        f"{len(missing)} selected images could not be located. "
        "Complete/check the uploads before continuing."
    )

# 5. Create the metadata table and import records
columns = [
    "dataset", "image_id", "lesion_id", "patient_id",
    "original_label", "class_label", "age", "sex",
    "localization", "image_path"
]

# Convert missing values to SQL NULL and NumPy scalars to Python values.
def sql_value(value):
    if pd.isna(value):
        return None
    return value.item() if hasattr(value, "item") else value

rows = [
    tuple(sql_value(value) for value in row)
    for row in records[columns].itertuples(index=False, name=None)
]

with sqlite3.connect(DB_PATH) as conn:
    conn.execute("""
        CREATE TABLE IF NOT EXISTS images (
            id INTEGER PRIMARY KEY,
            dataset TEXT NOT NULL
                CHECK (dataset IN ('HAM10000', 'PAD-UFES-20')),
            image_id TEXT NOT NULL,
            lesion_id TEXT,
            patient_id TEXT,
            original_label TEXT NOT NULL,
            class_label TEXT NOT NULL
                CHECK (class_label IN ('Melanoma', 'BCC', 'SCC')),
            age REAL,
            sex TEXT,
            localization TEXT,
            image_path TEXT NOT NULL,
            UNIQUE(dataset, image_id)
        )
    """)

    conn.executemany("""
        INSERT INTO images (
            dataset, image_id, lesion_id, patient_id,
            original_label, class_label, age, sex,
            localization, image_path
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT(dataset, image_id) DO UPDATE SET
            lesion_id = excluded.lesion_id,
            patient_id = excluded.patient_id,
            original_label = excluded.original_label,
            class_label = excluded.class_label,
            age = excluded.age,
            sex = excluded.sex,
            localization = excluded.localization,
            image_path = excluded.image_path
    """, rows)

    conn.execute(
        "CREATE INDEX IF NOT EXISTS idx_images_class "
        "ON images(class_label)"
    )

    total = conn.execute("SELECT COUNT(*) FROM images").fetchone()[0]
    if total != len(records):
        raise ValueError(
            f"Database contains {total} records, but current metadata "
            f"selects {len(records)}. Check for older records."
        )

# 6. Verify the saved database
with sqlite3.connect(DB_PATH) as conn:
    integrity = conn.execute("PRAGMA integrity_check").fetchone()[0]
    if integrity != "ok":
        raise RuntimeError(f"Database integrity check failed: {integrity}")

    print("\nRecords by dataset and class:")
    display(pd.read_sql_query("""
        SELECT dataset, class_label, COUNT(*) AS image_count
        FROM images
        GROUP BY dataset, class_label
        ORDER BY dataset, class_label
    """, conn))

    print("Combined class distribution:")
    display(pd.read_sql_query("""
        SELECT class_label, COUNT(*) AS image_count
        FROM images
        GROUP BY class_label
        ORDER BY class_label
    """, conn))

    print("Sample database records:")
    display(pd.read_sql_query(
        "SELECT * FROM images LIMIT 5", conn
    ))

print("\nDATABASE SETUP COMPLETE")
print("Total records:", total)
print("Missing selected image files:", len(missing))
print("Database integrity:", integrity)
print("Database saved to:", DB_PATH)

Locating images. This may take a few minutes...

Records by dataset and class:


,dataset,class_label,image_count
0,HAM10000,BCC,514
1,HAM10000,Melanoma,1113
2,PAD-UFES-20,BCC,845
3,PAD-UFES-20,Melanoma,52
4,PAD-UFES-20,SCC,192


Combined class distribution:


,class_label,image_count
0,BCC,1359
1,Melanoma,1165
2,SCC,192


Sample database records:


,id,dataset,image_id,lesion_id,patient_id,original_label,class_label,age,sex,localization,image_path
0,1,HAM10000,ISIC_0025964,HAM_0000871,None,mel,Melanoma,40.0,female,chest,/content/drive/MyDrive/cnn-skin-cancer-classif...
1,2,HAM10000,ISIC_0030623,HAM_0000871,None,mel,Melanoma,40.0,female,chest,/content/drive/MyDrive/cnn-skin-cancer-classif...
2,3,HAM10000,ISIC_0027190,HAM_0000040,None,mel,Melanoma,80.0,male,upper extremity,/content/drive/MyDrive/cnn-skin-cancer-classif...
3,4,HAM10000,ISIC_0031023,HAM_0005678,None,mel,Melanoma,60.0,male,chest,/content/drive/MyDrive/cnn-skin-cancer-classif...
4,5,HAM10000,ISIC_0028086,HAM_0005678,None,mel,Melanoma,60.0,male,chest,/content/drive/MyDrive/cnn-skin-cancer-classif...



DATABASE SETUP COMPLETE
Total records: 2716
Missing selected image files: 0
Database integrity: ok
Database saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Database/skin_cancer.db
